In [1]:
import os

# Define the root directory name
PROJECT_NAME = "hair-ai-trichologist"

# Define the folder hierarchy
STRUCTURE = [
    f"{PROJECT_NAME}/data/raw",
    f"{PROJECT_NAME}/data/processed",
    f"{PROJECT_NAME}/notebooks",
    f"{PROJECT_NAME}/src",
    f"{PROJECT_NAME}/reports/figures"
]

# Create directories
print("Creating project directory structure...")
for folder in STRUCTURE:
    os.makedirs(folder, exist_ok=True)
    print(f" -> Created: {folder}")

# Create empty placeholder files to make modules importable later
files_to_touch = [
    f"{PROJECT_NAME}/src/__init__.py",
    f"{PROJECT_NAME}/README.md",
    f"{PROJECT_NAME}/requirements.txt"
]

for file_path in files_to_touch:
    if not os.path.exists(file_path):
        with open(file_path, 'w') as f:
            if "README" in file_path:
                f.write(f"# {PROJECT_NAME.replace('-', ' ').title()}\nAn AI-powered Trichologist and Hair Care Ingredient Formulator.")
            else:
                f.write("")
        print(f" -> Created file: {file_path}")

print("\nWorkspace setup complete.")

Creating project directory structure...
 -> Created: hair-ai-trichologist/data/raw
 -> Created: hair-ai-trichologist/data/processed
 -> Created: hair-ai-trichologist/notebooks
 -> Created: hair-ai-trichologist/src
 -> Created: hair-ai-trichologist/reports/figures
 -> Created file: hair-ai-trichologist/src/__init__.py
 -> Created file: hair-ai-trichologist/README.md
 -> Created file: hair-ai-trichologist/requirements.txt

Workspace setup complete.


fetching data and saving EDA plots

In [6]:
import os
import shutil
import kagglehub
import matplotlib.pyplot as plt

# Define our destination paths inside our repository architecture
TARGET_RAW_DIR = "hair-ai-trichologist/data/raw/hair_images"
FIGURES_PATH = "hair-ai-trichologist/reports/figures"

print("Downloading Real Dataset via kagglehub ---")
downloaded_path = kagglehub.dataset_download("kavyasreeb/hair-type-dataset")
print(f"Downloaded source files located at: {downloaded_path}")

# Clean up any existing raw image folder
if os.path.exists(TARGET_RAW_DIR):
    shutil.rmtree(TARGET_RAW_DIR)

# Move the downloaded dataset into our local repository structure
shutil.copytree(downloaded_path, TARGET_RAW_DIR)

# Check if kagglehub nested everything inside an extra 'data' folder
nested_data_dir = os.path.join(TARGET_RAW_DIR, "data")
if os.path.exists(nested_data_dir):
    print("-> Detected nested 'data' folder. Extracting classes to root raw directory...")
    # Move all subfolders out of 'data' up to TARGET_RAW_DIR
    for item in os.listdir(nested_data_dir):
        s_item = os.path.join(nested_data_dir, item)
        d_item = os.path.join(TARGET_RAW_DIR, item)
        shutil.move(s_item, d_item)
    # Remove the now empty nested 'data' directory
    os.rmdir(nested_data_dir)

print(f"Successfully moved and restructured real dataset into: {TARGET_RAW_DIR}\n")

print("Generating Distribution Chart for GitHub ---")
class_counts = {}
for item in os.listdir(TARGET_RAW_DIR):
    item_path = os.path.join(TARGET_RAW_DIR, item)
    if os.path.isdir(item_path):
        valid_files = [f for f in os.listdir(item_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if valid_files:
            class_counts[item] = len(valid_files)

# Plot and save the distribution chart
plt.figure(figsize=(8, 5))
plt.bar(class_counts.keys(), class_counts.values(), color='#4A90E2', edgecolor='black')
plt.title('Real Hair Dataset Class Distribution')
plt.xlabel('Extracted Hair Categories')
plt.ylabel('Number of Images')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

distribution_plot_path = os.path.join(FIGURES_PATH, 'real_hair_distribution.png')
plt.savefig(distribution_plot_path, dpi=300, bbox_inches='tight')
plt.close()
print(f" -> Exported class distribution chart to: {distribution_plot_path}")

Using Colab cache for faster access to the 'hair-type-dataset' dataset.
Downloaded source files located at: /kaggle/input/hair-type-dataset
-> Detected nested 'data' folder. Extracting classes to root raw directory...
Successfully moved and restructured real dataset into: hair-ai-trichologist/data/raw/hair_images

Generating Distribution Chart for GitHub ---
 -> Exported class distribution chart to: hair-ai-trichologist/reports/figures/real_hair_distribution.png


saving sample grid images

In [7]:
import cv2

print("Visualizing Sample Images ---")

fig, axes = plt.subplots(1, len(class_counts), figsize=(15, 5))
subfolders = [d for d in os.listdir(TARGET_RAW_DIR) if os.path.isdir(os.path.join(TARGET_RAW_DIR, d))]

for idx, folder in enumerate(subfolders):
    folder_path = os.path.join(TARGET_RAW_DIR, folder)
    img_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    if img_files:
        sample_img_path = os.path.join(folder_path, img_files[0])
        img = cv2.imread(sample_img_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        axes[idx].imshow(img_rgb)
        axes[idx].set_title(f"Class: {folder}\n({img_rgb.shape[1]}x{img_rgb.shape[0]})")
        axes[idx].axis('off')

plt.tight_layout()
sample_grid_path = os.path.join(FIGURES_PATH, 'dataset_samples_grid.png')
plt.savefig(sample_grid_path, dpi=300, bbox_inches='tight')
plt.close()
print(f" -> Generated and saved dataset visual sample grid to: {sample_grid_path}")

Visualizing Sample Images ---
 -> Generated and saved dataset visual sample grid to: hair-ai-trichologist/reports/figures/dataset_samples_grid.png


In [4]:
%%writefile hair-ai-trichologist/src/data_check.py
import os

def verify_dataset_structure(image_dir):
    """
    Crawls our raw image directory to verify that the dataset is present,
    properly organized, and free of empty folders.
    """
    print(f"Running Data Check on: {image_dir} ---")

    if not os.path.exists(image_dir):
        print(f"[ERROR] Directory {image_dir} does not exist!")
        return False

    categories = [d for d in os.listdir(image_dir) if os.path.isdir(os.path.join(image_dir, d))]

    if not categories:
        print("[ERROR] No subdirectories/classes found in the image path.")
        return False

    print(f"[SUCCESS] Found {len(categories)} distinct hair classification categories.")
    for cat in categories:
        cat_path = os.path.join(image_dir, cat)
        files = [f for f in os.listdir(cat_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        print(f" -> Class '{cat}': Contains {len(files)} valid images.")

    print("Data validation process complete.\n")
    return True

if __name__ == "__main__":
    RAW_DIR = "hair-ai-trichologist/data/raw/hair_images"
    verify_dataset_structure(RAW_DIR)

Writing hair-ai-trichologist/src/data_check.py


In [8]:
!python hair-ai-trichologist/src/data_check.py

Running Data Check on: hair-ai-trichologist/data/raw/hair_images ---
[SUCCESS] Found 5 distinct hair classification categories.
 -> Class 'dreadlocks': Contains 443 valid images.
 -> Class 'curly': Contains 513 valid images.
 -> Class 'Wavy': Contains 329 valid images.
 -> Class 'Straight': Contains 485 valid images.
 -> Class 'kinky': Contains 217 valid images.
Data validation process complete.



In trichology, local environmental data (specifically relative humidity and dew point) dictates how hair strands behave. High humidity swells the hair shaft and breaks hydrogen bonds, causing frizz in porous hair (Curly/Kinky), while dry air saps moisture from it.

Let's use Open-Meteo since it works worldwide without an API key!

In [9]:
# This module that takes latitude and longitude coordinates (or simple location names),
# fetches current humidity and temperature, and translates that into a "Frizz & Moisture Risk Factor

%%writefile hair-ai-trichologist/src/weather_handler.py
import requests
import json

def fetch_local_weather(lat=40.7128, lon=-74.0060):
    """
    Fetches real-time relative humidity and temperature using the free Open-Meteo API.
    Defaults to New York City coordinates if none are provided.
    """
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m,relative_humidity_2m"

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status() # Raise exception for bad status codes
        data = response.json()

        current_stats = data.get("current", {})
        temperature = current_stats.get("temperature_2m")
        humidity = current_stats.get("relative_humidity_2m")

        return {
            "status": "success",
            "temperature_c": temperature,
            "relative_humidity_pct": humidity
        }
    except Exception as e:
        return {
            "status": "error",
            "message": str(e)
        }

def calculate_hair_environmental_risk(humidity_pct):
    """
    Translates raw humidity percentages into actionable clinical hair stress metrics.
    """
    if humidity_pct is None:
        return "Unknown", "Unable to calculate risk factors due to missing weather metrics."

    if humidity_pct >= 75:
        risk_level = "High Frizz / High Hydration Loss Risk"
        advice = "Humid air will break hydrogen bonds. Highly porous hair types (Curly/Kinky) require anti-humectants, heavy sealants, or silicones to lock out atmospheric moisture."
    elif 40 <= humidity_pct < 75:
        risk_level = "Optimal Optimal Balance"
        advice = "Atmospheric moisture is stable. Standard balanced routine of humectants (glycerin) and light emollients will perform perfectly."
    else:
        risk_level = "Severe Dryness / Dehydration Risk"
        advice = "Low humidity will strip moisture out of the hair shaft. Focus heavily on deep conditioning, leave-in creams, and water-soluble humectants."

    return risk_level, advice

if __name__ == "__main__":
    # Test the module with default coordinates
    print("--- Testing Weather Handler Module ---")
    weather_data = fetch_local_weather()
    print(f"Raw API Response Snippet: {weather_data}")

    if weather_data["status"] == "success":
        rh = weather_data["relative_humidity_pct"]
        risk, tips = calculate_hair_environmental_risk(rh)
        print(f"\nCurrent Humidity: {rh}%")
        print(f"Calculated Risk Level: {risk}")
        print(f"Trichologist Advice: {tips}\n")

Writing hair-ai-trichologist/src/weather_handler.py


In [10]:
!python hair-ai-trichologist/src/weather_handler.py

--- Testing Weather Handler Module ---
Raw API Response Snippet: {'status': 'success', 'temperature_c': 12.3, 'relative_humidity_pct': 62}

Current Humidity: 62%
Calculated Risk Level: Optimal Optimal Balance
Trichologist Advice: Atmospheric moisture is stable. Standard balanced routine of humectants (glycerin) and light emollients will perform perfectly.



Next, the NLP Ingredient Parser

The purpose of this module is to accept a raw string of text (the ingredients pasted from the back of a shampoo or conditioner bottle) and extract key chemical groups that directly impact hair health based on porosity.

We will focus on mapping three crucial component categories:

* Sulfate Cleaners: High-strip surfactants (e.g., Sodium Lauryl Sulfate) that damage fragile, high-porosity coily/kinky hair.

* Heavy Silicones: Non-water-soluble coatings (e.g., Dimethicone) that create immediate shine but cause severe buildup unless washed with harsh sulfates.

* Humectants: Moisture-binding agents (e.g., Glycerin, Propylene Glycol) that interact heavily with the atmospheric humidity data we just fetched.

In [11]:
%%writefile hair-ai-trichologist/src/ingredient_parser.py
import re
import json

def parse_ingredients(ingredient_string):
    """
    Cleans a raw ingredient text string, tokenizes it by commas,
    and checks each ingredient against strict chemical definitions.
    """
    if not ingredient_string or not isinstance(ingredient_string, str):
        return {"error": "Invalid text input"}

    # Normalize text: convert to lowercase and remove trailing periods or brackets
    clean_str = ingredient_string.lower().strip()
    # Split by comma
    raw_ingredients = [ing.strip() for ing in re.split(r',\s*', clean_str) if ing.strip()]

    # Define clinical trichology classifications using regex patterns
    patterns = {
        "sulfates": [r".*sulfate.*", r".*sulfosuccinate.*"],
        "heavy_silicones": [r".*dimethicone$", r".*amodimethicone.*", r".*cyclomethicone.*"],
        "water_soluble_silicones": [r".*copolyol.*", r"^peg-.*dimethicone.*"],
        "humectants": [r".*glycerin.*", r".*glycol.*", r".*panthenol.*", r".*hyaluronic.*"]
    }

    # Initialize our analysis report structural mapping
    analysis_report = {
        "total_ingredients_detected": len(raw_ingredients),
        "detected_groups": {
            "sulfates": [],
            "heavy_silicones": [],
            "water_soluble_silicones": [],
            "humectants": []
        },
        "flags": []
    }

    # Match ingredients against our criteria
    for ing in raw_ingredients:
        for group, regex_list in patterns.items():
            for pattern in regex_list:
                if re.match(pattern, ing):
                    # Avoid adding duplicates if multiple regexes catch the same item
                    if ing not in analysis_report["detected_groups"][group]:
                        analysis_report["detected_groups"][group].append(ing)

    # Generate engineering triggers based on safety profiles
    if analysis_report["detected_groups"]["sulfates"]:
        analysis_report["flags"].append("STRIPPING_RISK: Contains harsh surfactants that degrade fragile hair lipids.")

    if analysis_report["detected_groups"]["heavy_silicones"] and not analysis_report["detected_groups"]["sulfates"]:
        analysis_report["flags"].append("BUILDUP_RISK: Contains heavy silicones without a matching sulfate cleanser to strip them away.")

    return analysis_report

if __name__ == "__main__":
    print("--- Testing Ingredient Parser Module ---")

    # Test sample simulating a typical supermarket moisturizing conditioner
    sample_bottle = "Water, Cetearyl Alcohol, Dimethicone, Glycerin, Behentrimonium Chloride, Sodium Lauryl Sulfate, Propylene Glycol, Fragrance"

    results = parse_ingredients(sample_bottle)
    print(json.dumps(results, indent=4))

Writing hair-ai-trichologist/src/ingredient_parser.py


In [12]:
!python hair-ai-trichologist/src/ingredient_parser.py

--- Testing Ingredient Parser Module ---
{
    "total_ingredients_detected": 8,
    "detected_groups": {
        "sulfates": [
            "sodium lauryl sulfate"
        ],
        "heavy_silicones": [
            "dimethicone"
        ],
        "water_soluble_silicones": [],
        "humectants": [
            "glycerin",
            "propylene glycol"
        ]
    },
    "flags": [
        "STRIPPING_RISK: Contains harsh surfactants that degrade fragile hair lipids."
    ]
}


It isolated sodium lauryl sulfate as a stripping risk, logged dimethicone under heavy silicones, and correctly swept up glycerin and propylene glycol as humectants.

Notice how the BUILDUP_RISK flag was smart enough not to trigger because a sulfate was present to wash that silicone out.

We are going to construct a deep learning script using PyTorch and Torchvision. We will uae EfficientNet-B0—a highly accurate yet lightweight model architecture that is perfectly optimized to train within Colab's free T4 GPU RAM limits.

Our script will load the 5 data classes, resize/normalize the images, configure a transfer learning pipeline, and print out training tracking metrics.

In [13]:
%%writefile hair-ai-trichologist/src/vision_classifier.py
import os
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

def get_data_transformers():
    """
    Defines image transformations. Standardizes image sizing for EfficientNet
    and applies mild data augmentation to prevent overfitting.
    """
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return train_transform

def build_model(num_classes):
    """
    Loads a pre-trained EfficientNet-B0, freezes its base feature extractor layers,
    and updates the final classification head to match our hair category counts.
    """
    # Load weights using modern Torchvision API
    weights = models.EfficientNet_B0_Weights.DEFAULT
    model = models.efficientnet_b0(weights=weights)

    # Freeze feature layers so we don't destroy pre-trained weights during short Colab training
    for param in model.parameters():
        param.requires_grad = False

    # Replace the classification head
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes)
    )
    return model

def run_dry_test(image_dir):
    """
    Performs a rapid validation check to ensure PyTorch can load the classes
    and construct the network architecture without crashing.
    """
    print("--- Vision Module: Running Architecture Test ---")
    if not os.path.exists(image_dir):
        print(f"[ERROR] Cannot run dry test. Path missing: {image_dir}")
        return

    transform = get_data_transformers()
    try:
        dataset = datasets.ImageFolder(root=image_dir, transform=transform)
        print(f"[SUCCESS] Dataset successfully loaded by PyTorch ImageFolder wrapper.")
        print(f"Detected Classes: {dataset.classes}")

        # Instantiate model
        model = build_model(len(dataset.classes))
        print(f"[SUCCESS] EfficientNet-B0 loaded. Classification head configured for {len(dataset.classes)} target outputs.")

        # Check for GPU accessibility
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Active Training Hardware Target: {device.type.upper()}")

    except Exception as e:
        print(f"[ERROR] Failed dry check: {str(e)}")

if __name__ == "__main__":
    RAW_DIR = "hair-ai-trichologist/data/raw/hair_images"
    run_dry_test(RAW_DIR)

Writing hair-ai-trichologist/src/vision_classifier.py


In [14]:
!python hair-ai-trichologist/src/vision_classifier.py

--- Vision Module: Running Architecture Test ---
[SUCCESS] Dataset successfully loaded by PyTorch ImageFolder wrapper.
Detected Classes: ['Straight', 'Wavy', 'curly', 'dreadlocks', 'kinky']
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100% 20.5M/20.5M [00:00<00:00, 139MB/s]
[SUCCESS] EfficientNet-B0 loaded. Classification head configured for 5 target outputs.
Active Training Hardware Target: CUDA


In [17]:
%%writefile hair-ai-trichologist/src/main_orchestrator.py
import os
import random
import torch
from PIL import Image

# Import the exact modules we built in the previous steps
from weather_handler import fetch_local_weather, calculate_hair_environmental_risk
from ingredient_parser import parse_ingredients
from vision_classifier import get_data_transformers, build_model

def run_complete_system_test(image_path, ingredient_txt, lat=40.7128, lon=-74.0060):
    """
    Ties the Vision Model, NLP Parser, and Weather API together to output
    a unified, context-aware hair care assessment.
    """
    print("=" * 60)
    print("      Hair AI Trichologist")
    print("=" * 60)

    # --- 1. TEST WEATHER MODULE ---
    print("\n[STEP 1/3] Fetching Environmental Context...")
    weather = fetch_local_weather(lat, lon)
    if weather["status"] == "success":
        humidity = weather["relative_humidity_pct"]
        temp = weather["temperature_c"]
        env_risk, env_advice = calculate_hair_environmental_risk(humidity)
        print(f" -> Location Metrics: {temp}°C, {humidity}% Relative Humidity")
        print(f" -> Climate Threat Assessment: {env_risk}")
    else:
        env_advice = "Weather service unavailable. Defaulting to baseline product analysis."
        print(" -> [WARNING] Weather API fetch failed. Using safety defaults.")

    # --- 2. TEST INGREDIENT NLP PARSER ---
    print("\n[STEP 2/3] Analyzing Product Ingredient Formulation...")
    ingredient_results = parse_ingredients(ingredient_txt)
    print(f" -> Scanned {ingredient_results['total_ingredients_detected']} compounds.")
    for flag in ingredient_results["flags"]:
        print(f" -> [FLAGGED] {flag}")
    if not ingredient_results["flags"]:
        print(" -> No immediate formulation ingredient conflicts detected.")

    # --- 3. TEST VISION MODEL (MOCK INFERENCE LAYER) ---
    print("\n[STEP 3/3] Running Computer Vision Hair Diagnostics...")
    # Extract the true class from the folder name to verify accuracy
    true_class = os.path.basename(os.path.dirname(image_path))

    # Prepare image using our production transformations
    transform = get_data_transformers()
    try:
        img = Image.open(image_path).convert('RGB')
        img_tensor = transform(img).unsqueeze(0) # Add batch dimension

        # Simulating model forward pass prediction for testing purposes
        # (Since we haven't run the multi-hour training loop yet, we simulate the output tensor mapping)
        classes_map = ['Straight', 'Wavy', 'curly', 'dreadlocks', 'kinky']

        # Let's write a deterministic test logic: 85% chance to predict perfectly for the test run
        if random.random() > 0.15:
            predicted_class = true_class
            confidence = random.uniform(88.5, 99.2)
        else:
            predicted_class = random.choice(classes_map)
            confidence = random.uniform(50.0, 75.0)

        print(f" -> Image Source: {image_path}")
        print(f" -> Real Target Category: {true_class}")
        print(f" -> AI Predicted Category: {predicted_class} ({confidence:.2f}% Confidence)")

    except Exception as e:
        print(f" -> [ERROR] Vision inference pipeline failed: {str(e)}")
        predicted_class = "Unknown"

    # --- 4. THE INTEGRATED TRICHOLOGY OUTPUT ---
    print("\n" + "=" * 60)
    print("                 FINAL TRICHOLOGIST VERDICT")
    print("=" * 60)
    print(f"Diagnosed Profile : {predicted_class} Hair Texture Type")
    print(f"Environmental Risk: {env_advice}")

    # Dynamic logic combining hair type + ingredient flags
    if predicted_class.lower() in ['curly', 'kinky', 'dreadlocks'] and ingredient_results["detected_groups"]["sulfates"]:
        print("\nCRITICAL ROUTINE ADJUSTMENT REQUIRED:")
        print(" -> Because your hair type is highly prone to structural dryness, using the detected 'Sodium Lauryl Sulfate' surfactant in this climate will severely dehydrate your curls. Switch to a sulfate-free co-wash immediately.")
    else:
        print("\nROUTINE VERDICT:")
        print(" -> Your product formulation is structurally compatible with your diagnosed hair shape under current atmospheric conditions.")
    print("=" * 60 + "\n")

if __name__ == "__main__":
    # Automatically locate a valid random image from the downloaded dataset to use for our test
    IMAGE_DIR = "hair-ai-trichologist/data/raw/hair_images"
    all_imgs = []
    for root, _, files in os.walk(IMAGE_DIR):
        for f in files:
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                all_imgs.append(os.path.join(root, f))

    if not all_imgs:
        print("[ERROR] No images found to run test. Please ensure Step 2 ran successfully.")
    else:
        sample_image = random.choice(all_imgs)
        # Paste a sample shampoo ingredient text to test our NLP engine
        sample_ingredients = "Water, Sodium Lauryl Sulfate, Cocamidopropyl Betaine, Dimethicone, Glycerin, Phenoxyethanol"

        # Execute test (Using default NYC coordinates, but you can change these to your lat/lon!)
        run_complete_system_test(image_path=sample_image, ingredient_txt=sample_ingredients, lat=43.6532, lon=-79.3832)

Overwriting hair-ai-trichologist/src/main_orchestrator.py


In [18]:
!python hair-ai-trichologist/src/main_orchestrator.py

      Hair AI Trichologist

[STEP 1/3] Fetching Environmental Context...
 -> Location Metrics: 14.0°C, 59% Relative Humidity
 -> Climate Threat Assessment: Optimal Optimal Balance

[STEP 2/3] Analyzing Product Ingredient Formulation...
 -> Scanned 6 compounds.
 -> [FLAGGED] STRIPPING_RISK: Contains harsh surfactants that degrade fragile hair lipids.

[STEP 3/3] Running Computer Vision Hair Diagnostics...
 -> Image Source: hair-ai-trichologist/data/raw/hair_images/dreadlocks/image267.jpg
 -> Real Target Category: dreadlocks
 -> AI Predicted Category: dreadlocks (91.96% Confidence)

                 FINAL TRICHOLOGIST VERDICT
Diagnosed Profile : dreadlocks Hair Texture Type
Environmental Risk: Atmospheric moisture is stable. Standard balanced routine of humectants (glycerin) and light emollients will perform perfectly.

CRITICAL ROUTINE ADJUSTMENT REQUIRED:
 -> Because your hair type is highly prone to structural dryness, using the detected 'Sodium Lauryl Sulfate' surfactant in this clim

In [19]:
!pip install -q gradio

In [24]:
%%writefile hair-ai-trichologist/src/app_interface.py
import os
import torch
import gradio as gr
from PIL import Image
from torchvision import transforms

# Import our production helper modules
from weather_handler import fetch_local_weather, calculate_hair_environmental_risk
from ingredient_parser import parse_ingredients
from vision_classifier import get_data_transformers, build_model

# Global configurations
MODEL_WEIGHTS_PATH = "hair-ai-trichologist/data/processed/efficientnet_hair_weights.pth"
CLASSES_PATH = "hair-ai-trichologist/src/classes.txt"

# Load the class labels dynamically
if os.path.exists(CLASSES_PATH):
    with open(CLASSES_PATH, "r") as f:
        CLASSES = [line.strip() for line in f.readlines()]
else:
    CLASSES = ['Straight', 'Wavy', 'curly', 'dreadlocks', 'kinky'] # Fallback default

# Instantiate and prepare the real model globally so it stays warm in memory
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL = None

if os.path.exists(MODEL_WEIGHTS_PATH):
    print(f"-> Loading trained model weights into app interface memory ({DEVICE.type.upper()})...")
    MODEL = build_model(num_classes=len(CLASSES))
    MODEL.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=DEVICE))
    MODEL.to(DEVICE)
    MODEL.eval()
else:
    print("-> [WARNING] No trained model weights found. App will run in simulation mode.")

def UI_orchestrator(input_image, zipcode_coords, ingredient_text):
    """
    The core bridge connecting visual UI inputs to our real PyTorch model weights
    and secondary textual APIs.
    """
    if input_image is None:
        return "⚠️ Error: Please upload or take a photo of your hair/scalp first!", "", "", ""

    # Parse coordinates from user input (Default to Toronto if blank/malformed)
    lat, lon = 43.6532, -79.3832
    if zipcode_coords and "," in zipcode_coords:
        try:
            lat = float(zipcode_coords.split(",")[0].strip())
            lon = float(zipcode_coords.split(",")[1].strip())
        except:
            pass

    # --- 1. Weather Logic ---
    weather = fetch_local_weather(lat, lon)
    if weather["status"] == "success":
        humidity = weather["relative_humidity_pct"]
        temp = weather["temperature_c"]
        env_risk, env_advice = calculate_hair_environmental_risk(humidity)
        weather_output = f"🌡️ Temperature: {temp}°C\n💧 Relative Humidity: {humidity}%\n⚠️ Climate Risk: {env_risk}"
    else:
        env_advice = "Weather service data unavailable."
        weather_output = "❌ Weather API Fetch Failed."

    # --- 2. Ingredient Logic ---
    ingredient_results = parse_ingredients(ingredient_text)
    flags_list = "\n".join([f"• {f}" for f in ingredient_results["flags"]])
    if not flags_list:
        flags_list = "• No immediate formulation conflicts detected."

    ingredient_output = (
        f"📋 Total Compounds Detected: {ingredient_results['total_ingredients_detected']}\n"
        f"🚨 Formulation Flags:\n{flags_list}"
    )

    # --- 3. Real Vision Inference Logic ---
    if MODEL is not None:
        try:
            # Open image file and run our standardized transformations
            img = Image.open(input_image).convert('RGB')
            transform = get_data_transformers()
            img_tensor = transform(img).unsqueeze(0).to(DEVICE) # Add batch dimension and push to GPU

            with torch.no_grad():
                outputs = MODEL(img_tensor)
                probabilities = torch.nn.functional.softmax(outputs[0], dim=0)
                confidence, class_idx = torch.max(probabilities, 0)

            predicted_class = CLASSES[class_idx.item()]
            confidence_score = confidence.item() * 100
            vision_output = f"🧬 Diagnosed Texture: {predicted_class}\n🎯 AI Confidence: {confidence_score:.2f}%"
        except Exception as e:
            predicted_class = "Unknown"
            vision_output = f"❌ Vision Engine Failure: {str(e)}"
            confidence_score = 0.0
    else:
        # Fallback if model files aren't found
        predicted_class = "Straight"
        vision_output = "ℹ️ Running in Simulation Mode (No weights detected)."

    # --- 4. Generate Unified Clinical Decision ---
    verdict = (
        f"==================================================\n"
        f"               TRICHOLOGIST CLINICAL VERDICT       \n"
        f"==================================================\n"
        f"Diagnosed Profile: {predicted_class} Hair Texture Type\n\n"
        f"Environmental Advice:\n{env_advice}\n\n"
    )

    if predicted_class.lower() in ['curly', 'kinky', 'dreadlocks'] and ingredient_results["detected_groups"]["sulfates"]:
        verdict += (
            "🚨 CRITICAL ROUTINE ADJUSTMENT REQUIRED:\n"
            "Because your hair type is structurally prone to severe moisture loss, using a product containing harsh "
            "cleansing sulfates in this specific climate will strip your protective lipid layers. "
            "Switch to a gentle, sulfate-free cream co-wash immediately to protect your pattern."
        )
    else:
        verdict += "✅ ROUTINE VERDICT:\nYour product formulation is structurally compatible with your diagnosed hair shape under current atmospheric conditions."

    return vision_output, weather_output, ingredient_output, verdict

# --- GRADIO INTERFACE DESIGN ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 💇‍♂️ AI Trichologist & Ingredient Formulator")
    gr.Markdown("Identify your real hair profile, evaluate chemical ingredients, and analyze real-time climate impacts instantly.")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 📥 Step 1: User Inputs")
            input_img = gr.Image(type="filepath", label="Upload Hair/Scalp Photo or Use Camera")
            geo_input = gr.Textbox(value="43.6532, -79.3832", label="Location Coordinates (Lat, Lon)", placeholder="e.g., 40.7128, -74.0060")
            ing_input = gr.Textbox(
                value="Water, Sodium Lauryl Sulfate, Dimethicone, Glycerin",
                label="Product Ingredients (Paste text list separated by commas)",
                lines=3
            )
            submit_btn = gr.Button("Analyze My Hair Profile", variant="primary")

        with gr.Column():
            gr.Markdown("### 📤 Step 2: System Diagnostic Metrics")
            with gr.Row():
                vis_out = gr.Textbox(label="Computer Vision Analysis", lines=2)
                wea_out = gr.Textbox(label="Local Weather Telemetry", lines=3)
            ing_out = gr.Textbox(label="Ingredient Analysis Summary", lines=3)

    gr.Markdown("### 📋 Step 3: Comprehensive Recommendation")
    final_verdict = gr.Textbox(label="AI Trichologist Output Report", lines=8)

    submit_btn.click(
        fn=UI_orchestrator,
        inputs=[input_img, geo_input, ing_input],
        outputs=[vis_out, wea_out, ing_out, final_verdict]
    )

if __name__ == "__main__":
    demo.launch(share=True, debug=True)

Overwriting hair-ai-trichologist/src/app_interface.py


In [25]:
!python hair-ai-trichologist/src/app_interface.py

-> Loading trained model weights into app interface memory (CUDA)...
/content/hair-ai-trichologist/src/app_interface.py:122: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://a86568a3e334acac5e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 3043, in block_thread
    time.sleep(0.1)
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/hai

In [22]:
%%writefile hair-ai-trichologist/src/train_vision_model.py
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

# Import configurations directly from our vision classifier module
from vision_classifier import get_data_transformers, build_model

def train_hair_classifier(data_dir, output_model_path, figures_dir, epochs=3, batch_size=32):
    print("=" * 60)
    print("          STARTING AI HAIR CLASSIFIER TRAINING")
    print("=" * 60)

    # Set device engine (Lock onto Colab's T4 GPU)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f" -> Execution Device: {device.type.upper()}")

    # 1. Prepare Datasets and DataLoaders
    transform = get_data_transformers()
    full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)
    num_classes = len(full_dataset.classes)

    # Save class names as a text file for our live UI inference engine to read later
    with open("hair-ai-trichologist/src/classes.txt", "w") as f:
        f.write("\n".join(full_dataset.classes))

    # Split into 80% Train, 20% Validation
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    print(f" -> Total Images Found: {len(full_dataset)} across {num_classes} categories.")
    print(f" -> Training Allocation: {len(train_dataset)} | Validation Allocation: {len(val_dataset)}")

    # 2. Instantiate Model, Loss Function, and Optimizer
    model = build_model(num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    # Only optimize parameters of our custom classification head (since the base is frozen)
    optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

    # Tracking logs for plotting
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    # 3. Core Training Loop
    for epoch in range(epochs):
        start_time = time.time()
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        epoch_train_loss = running_loss / len(train_dataset)

        # Validation evaluation
        model.eval()
        val_loss = 0.0
        corrects = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)

                _, preds = torch.max(outputs, 1)
                corrects += torch.sum(preds == labels.data)

        epoch_val_loss = val_loss / len(val_dataset)
        epoch_val_acc = corrects.double() / len(val_dataset)

        # Save historical metrics
        history["train_loss"].append(epoch_train_loss)
        history["val_loss"].append(epoch_val_loss)
        history["val_acc"].append(epoch_val_acc.item())

        duration = time.time() - start_time
        print(f"Epoch {epoch+1}/{epochs} ({duration:.1f}s) -> Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")

    # 4. Save Trained Model Weights
    os.makedirs(os.path.dirname(output_model_path), exist_ok=True)
    torch.save(model.state_dict(), output_model_path)
    print(f"\n[SUCCESS] Model weights saved securely to: {output_model_path}")

    # 5. Plot and Export Training Diagnostics
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(range(1, epochs+1), history["train_loss"], label="Train Loss", marker='o')
    plt.plot(range(1, epochs+1), history["val_loss"], label="Val Loss", marker='s')
    plt.title("Model Cross-Entropy Loss Convergence")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)

    plt.subplot(1, 2, 2)
    plt.plot(range(1, epochs+1), history["val_acc"], label="Validation Accuracy", color="green", marker='^')
    plt.title("Model Validation Accuracy Performance")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy Score")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)

    plt.tight_layout()
    curve_plot_path = os.path.join(figures_dir, "training_metrics_curves.png")
    plt.savefig(curve_plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SUCCESS] Training convergence visual graphics saved to: {curve_plot_path}\n")

if __name__ == "__main__":
    RAW_DIR = "hair-ai-trichologist/data/raw/hair_images"
    MODEL_OUT = "hair-ai-trichologist/data/processed/efficientnet_hair_weights.pth"
    FIGS_OUT = "hair-ai-trichologist/reports/figures"

    # Running for a lightweight 3 epochs since the base network features are frozen.
    # This runs comfortably inside Google Colab free tier limits in just a few minutes!
    train_hair_classifier(data_dir=RAW_DIR, output_model_path=MODEL_OUT, figures_dir=FIGS_OUT, epochs=3)

Writing hair-ai-trichologist/src/train_vision_model.py


In [23]:
!python hair-ai-trichologist/src/train_vision_model.py

          STARTING AI HAIR CLASSIFIER TRAINING
 -> Execution Device: CUDA
 -> Total Images Found: 1991 across 5 categories.
 -> Training Allocation: 1592 | Validation Allocation: 399
Epoch 1/3 (19.7s) -> Train Loss: 0.9577 | Val Loss: 0.5389 | Val Acc: 0.7995
Epoch 2/3 (17.1s) -> Train Loss: 0.5585 | Val Loss: 0.5063 | Val Acc: 0.8120
Epoch 3/3 (17.2s) -> Train Loss: 0.4817 | Val Loss: 0.4453 | Val Acc: 0.8496

[SUCCESS] Model weights saved securely to: hair-ai-trichologist/data/processed/efficientnet_hair_weights.pth
[SUCCESS] Training convergence visual graphics saved to: hair-ai-trichologist/reports/figures/training_metrics_curves.png

